# Clase 5 — Práctica: clasificador binario cuántico

Ya entrenaste el clasificador binario del ejemplo con el operador de lectura `cirq.X(qubit)`. En esta práctica cambias la base de medida y observas cómo afecta al aprendizaje del modelo.

## Enunciado

Cambia el operador de lectura de `cirq.X(qubit)` a `cirq.Z(qubit)` y vuelve a entrenar el modelo desde cero. Compara la curva de precisión de validación con la del ejemplo original.

Si el modelo no aprende igual de bien, piensa por qué: recuerda que los datos se codificaron con una rotación en el eje X (`cirq.X(qubit)**angulo`), así que la base de medida más informativa no tiene por qué ser la misma que la base de codificación.

**Pistas**

- Reutiliza `generar_datos`, `construir_circuito` y `precision_hinge` tal cual
- Solo cambia la lista `operador_lectura`
- Crea el modelo, la capa `PQC` y el `compile` de nuevo (un modelo nuevo, no reentrenes el anterior)
- Guarda el historial en una variable distinta para poder comparar las dos curvas en la misma gráfica

In [ ]:
import tensorflow_quantum as tfq
import tensorflow as tf
import cirq
import sympy
import random
import numpy as np
import matplotlib.pyplot as plt


def generar_datos(n_entrenamiento, n_prueba):
    """Genera ángulos cercanos a 0 (etiqueta +1) o cercanos a 1 (etiqueta -1)."""
    qubit = cirq.GridQubit(0, 0)

    def lote(n):
        circuitos, etiquetas = [], []
        for _ in range(n):
            angulo = random.uniform(0, 0.1) if random.random() < 0.5 else random.uniform(0.9, 1)
            circuito = cirq.Circuit()
            circuito.append(cirq.X(qubit) ** angulo)
            circuitos.append(circuito)
            etiquetas.append(1 if angulo < 0.5 else -1)
        return tfq.convert_to_tensor(circuitos), np.array(etiquetas)

    entrenamiento, etiquetas_entrenamiento = lote(n_entrenamiento)
    prueba, etiquetas_prueba = lote(n_prueba)
    return entrenamiento, etiquetas_entrenamiento, prueba, etiquetas_prueba


def construir_circuito(qubit):
    x, y, z = sympy.symbols("rotacion_x rotacion_y rotacion_z")
    circuito = cirq.Circuit()
    circuito.append(cirq.rx(x).on(qubit))
    circuito.append(cirq.ry(y).on(qubit))
    circuito.append(cirq.rz(z).on(qubit))
    return circuito


def precision_hinge(etiqueta_real, prediccion):
    etiqueta_real = tf.squeeze(etiqueta_real) > 0.0
    prediccion = tf.squeeze(prediccion) > 0.0
    return tf.reduce_mean(tf.cast(etiqueta_real == prediccion, tf.float32))

In [ ]:
# Reutiliza los mismos datos para que la comparación entre las dos bases de medida sea justa
entrenamiento, etiquetas_entrenamiento, prueba, etiquetas_prueba = generar_datos(1000, 100)

qubit = cirq.GridQubit(0, 0)

# TODO: cambia el operador de lectura de cirq.X(qubit) a cirq.Z(qubit)
operador_lectura = [cirq.X(qubit)]

entradas = tf.keras.Input(shape=(), dtype=tf.dtypes.string)

# TODO: construye la capa tfq.layers.PQC con construir_circuito(qubit), operador_lectura,
# repetitions=32, differentiator=tfq.differentiators.ParameterShift() e
# initializer=tf.keras.initializers.Zeros, aplicada sobre 'entradas'
capa_cuantica = None

# TODO: crea el modelo de Keras (tf.keras.models.Model) con inputs=entradas y outputs=capa_cuantica
modelo_z = None

# TODO: compila el modelo con Adam(learning_rate=0.001), loss=tf.keras.losses.hinge
# y metrics=[precision_hinge]


In [ ]:
# TODO: entrena modelo_z con .fit(...) durante 64 épocas, batch_size=32 y
# validation_data=(prueba, etiquetas_prueba); guarda el resultado en historial_z
historial_z = None

# TODO: dibuja historial_z.history["precision_hinge"] y historial_z.history["val_precision_hinge"]
# junto con un título y leyenda que indiquen que este modelo usa cirq.Z(qubit) como lectura


<details>
<summary>💡 Ver solución</summary>

```python
entrenamiento, etiquetas_entrenamiento, prueba, etiquetas_prueba = generar_datos(1000, 100)

qubit = cirq.GridQubit(0, 0)

# Cambio clave: la lectura ahora es en la base Z en vez de la base X
operador_lectura = [cirq.Z(qubit)]

entradas = tf.keras.Input(shape=(), dtype=tf.dtypes.string)

capa_cuantica = tfq.layers.PQC(
    construir_circuito(qubit),
    operador_lectura,
    repetitions=32,
    differentiator=tfq.differentiators.ParameterShift(),
    initializer=tf.keras.initializers.Zeros,
)(entradas)

modelo_z = tf.keras.models.Model(inputs=entradas, outputs=capa_cuantica)
modelo_z.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.hinge,
    metrics=[precision_hinge],
)

historial_z = modelo_z.fit(
    entrenamiento,
    etiquetas_entrenamiento,
    epochs=64,
    batch_size=32,
    validation_data=(prueba, etiquetas_prueba),
)

plt.plot(historial_z.history["precision_hinge"], label="Entrenamiento (lectura Z)")
plt.plot(historial_z.history["val_precision_hinge"], label="Validación (lectura Z)")
plt.legend()
plt.title("Precisión del clasificador con lectura en base Z")
plt.xlabel("Épocas")
plt.ylabel("Precisión")
plt.show()

# Razonamiento: los datos se codifican con cirq.X(qubit)**angulo, una rotación alrededor
# del eje X. Esa rotación mueve el estado dentro del plano Y-Z de la esfera de Bloch,
# así que la información sobre el ángulo queda codificada sobre todo en las componentes
# Y y Z del estado, no en la componente X. Medir con cirq.Z(qubit) puede captar peor (o mejor,
# según el caso) esa información que medir con cirq.X(qubit), porque la base de medida
# no coincide con la base de codificación. Si la curva de precisión con lectura Z
# converge más despacio o se estanca por debajo del 100%, es una señal de que el circuito
# entrenable (rx, ry, rz) tiene que compensar ese desajuste ajustando sus tres rotaciones,
# lo que puede requerir más épocas o no converger igual de bien.
```

</details>